In [1]:
from pathlib import Path
import pandas as pd
import spacy

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"


In [2]:
df = pd.read_csv(DATA_DIR / "fake_job_postings_processed.csv")
df.shape


(17880, 23)

## spaCy Text Preprocessing
1. Load the cleaned text produced in the EDA notebook.
2. Use spaCy lemmatization with stop-word, punctuation, and whitespace removal.
3. Keep this notebook focused on preprocessing only, not modeling.
4. Fail fast if the spaCy model is missing, because saving a dataframe without `clean_text` would silently break downstream notebooks.


In [3]:
# spaCy pipeline
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError as exc:
    raise RuntimeError(
        "spaCy model en_core_web_sm is required. Run: python -m spacy download en_core_web_sm"
    ) from exc

def preprocess_doc(doc):
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and not token.is_space
    ]
    return " ".join(tokens)

print("Starting spaCy preprocessing... this may take a few minutes.")
truncated_text = (text[:100000] for text in df["text"].astype(str))
df["clean_text"] = [
    preprocess_doc(doc)
    for doc in nlp.pipe(truncated_text, batch_size=50)
]

empty_clean = (df["clean_text"].str.strip() == "").sum()
print(f"Rows with empty clean_text: {empty_clean}")
df[["text", "clean_text"]].head()


Starting spaCy preprocessing... this may take a few minutes.
Rows with empty clean_text: 0


,text,clean_text
0,marketing intern we re food52 and we ve create...,marketing intern food52 ve create groundbreaki...
1,customer service cloud video production 90 sec...,customer service cloud video production 90 sec...
2,commissioning machinery assistant cma valor se...,commission machinery assistant cma valor servi...
3,account executive washington dc our passion fo...,account executive washington dc passion improv...
4,bill review manager spotsource solutions llc i...,bill review manager spotsource solution llc gl...


In [4]:
# Quick quality check
df["clean_length"] = df["clean_text"].str.len()
df[["clean_length"]].describe()


,clean_length
count,17880.000000
mean,1880.434508
std,1045.175534
min,14.000000
25%,1092.000000
50%,1781.500000
75%,2468.000000
max,11086.000000


In [5]:
# Save cleaned data for modeling
if "clean_text" not in df.columns:
    raise RuntimeError("clean_text was not created; preprocessing failed.")

df.to_csv(DATA_DIR / "fake_job_postings_processed_1.csv", index=False)
